In [1]:
data = "data\\Train.csv"

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [3]:
df = pd.read_csv(data)

In [26]:
df.describe()

,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
count,14633.000000,14633.000000,14633.000000,14633.000000,14633.000000,14633.000000,14633.000000,14633.000000,14633.000000,14633.000000
mean,0.597370,-8.441599,0.083957,0.280623,0.136972,0.193121,0.523717,241743.824913,0.500550,118.968750
std,0.176921,4.727868,0.058226,0.238181,0.158300,0.135068,0.241670,67216.815488,0.288769,26.566071
min,0.076900,-27.509725,0.023500,0.000005,0.000001,0.024300,0.025600,63973.000000,0.000067,46.718000
25%,0.474349,-11.659427,0.023500,0.061750,0.000001,0.075404,0.351998,196588.623600,0.250600,100.888445
50%,0.600233,-8.236074,0.072629,0.252848,0.082805,0.180057,0.525871,241235.859000,0.500800,118.733631
75%,0.720912,-4.829767,0.122715,0.441652,0.236554,0.287646,0.699439,286987.805700,0.750600,136.728906
max,0.975000,-1.357000,0.346387,0.995000,0.890385,0.803157,0.978000,519650.691100,1.000000,206.037000


In [27]:
df.head()

,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
0,0.513080,-7.811659,0.071013,0.064564,0.109495,0.316042,0.736929,328639.3188,0.556200,117.092439
1,0.775393,-6.819409,0.023500,0.510599,0.187498,0.024361,0.259488,271967.9826,0.410533,122.002279
2,0.636408,-19.782248,0.063451,0.427861,0.002226,0.024300,0.054848,186147.0029,0.533333,149.130616
3,0.232190,-14.957299,0.023500,0.076268,0.000001,0.228454,0.744650,321734.9723,0.658533,95.832178
4,0.758564,-4.715966,0.023500,0.263551,0.414794,0.197167,0.966592,179973.3982,0.230467,125.696263


In [28]:
depth_range = [1,2,3,4,5,6,7,8,9]
minsplit_range = [2,3,4,5,10,20,25,30]
minleaf_range = [1,2,3,4,5,10,15]

max_leaf_nodes = [1,2,3,4,5,7,3,8,10,None]



parameters = dict(max_depth=depth_range,
                  min_samples_split=minsplit_range, 
                  min_samples_leaf=minleaf_range,
             
                 max_leaf_nodes = max_leaf_nodes
                )

In [29]:
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor
kfold = KFold(n_splits=5, random_state=2021,shuffle=True)
from sklearn.model_selection import GridSearchCV
clf = DecisionTreeRegressor(random_state=2021)
cv = GridSearchCV(clf, param_grid=parameters,
                  cv=kfold,scoring="neg_root_mean_squared_error")

In [30]:
# df = df.drop(['BeatsPerMinute'],axis=1)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state = 1)
df_train, df_val= train_test_split(df_full_train, test_size=0.25, random_state=1)

In [31]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_train = (df_train.BeatsPerMinute ).astype('int').values
y_val = (df_val.BeatsPerMinute ).astype('int').values
y_test = (df_test.BeatsPerMinute).astype('int').values

del df_train['BeatsPerMinute']
del df_test['BeatsPerMinute']
del df_val['BeatsPerMinute']

In [32]:
dv = DictVectorizer(sparse=True)
train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
val_dicts = df_val.to_dict(orient='records')
X_val= dv.fit_transform(val_dicts)

In [11]:
cv.fit(df_train,y_train)

C:\Users\AJAY\miniconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
2520 fits failed out of a total of 25200.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2520 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\AJAY\miniconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AJAY\miniconda3\Lib\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\AJAY\miniconda3\Lib\site-packages\sklearn\base.py", line 471, in _validate_para

,estimator,DecisionTreeR...om_state=2021)
,param_grid,"{'max_depth': [1, 2, ...], 'max_leaf_nodes': [1, 2, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 3, ...]}"
,scoring,'neg_root_mean_squared_error'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'squared_error'


In [12]:
print(cv.best_params_)

print(cv.best_score_)

print(cv.best_estimator_)

{'max_depth': 1, 'max_leaf_nodes': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}
-26.568633343874534
DecisionTreeRegressor(max_depth=1, max_leaf_nodes=2, random_state=2021)


In [33]:
clf = DecisionTreeRegressor( max_depth=1, max_leaf_nodes= 2,
                      min_samples_leaf=1, min_samples_split=2,
                      random_state=2021)


In [34]:
clf.fit(df_train,y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,2021
,max_leaf_nodes,2
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [35]:
y_pred1 = clf.predict(X_val)

C:\Users\AJAY\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(


In [36]:
y_pred1[y_pred1 <0] = 0

In [37]:
import numpy as np
rmse = np.sqrt(mean_squared_error(y_val, y_pred1))

In [38]:
print(rmse)

26.451312799979185


## Linear Regression

In [39]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)
y_val_pred = lr_model.predict(X_val)

mae = mean_absolute_error(y_val, y_val_pred)
mse = mean_squared_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"RMSE: {np.sqrt(mse):.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.2f}")



Mean Absolute Error (MAE): 21.08
RMSE: 26.41
Mean Squared Error (MSE): 697.51
R² Score: -0.00


In [40]:
feature_importance = pd.DataFrame({
    'Feature': dv.feature_names_,
    'Coefficient': lr_model.coef_ 
}).sort_values(by='Coefficient', ascending=False)
 
feature_importance['Coefficient'] = feature_importance['Coefficient'].round(3)
print('positive top10 features: ',feature_importance.head(10))


positive top10 features:                       Feature  Coefficient
1              AudioLoudness        0.080
5                  MoodScore        0.002
0            AcousticQuality        0.001
7            TrackDurationMs        0.000
4  LivePerformanceLikelihood       -0.000
8               VocalContent       -0.001
6                RhythmScore       -0.003
2                     Energy       -0.003
3          InstrumentalScore       -0.005


In [41]:
from sklearn.model_selection import cross_val_score
 
scores = cross_val_score(LinearRegression(), X_train, y_train, scoring='neg_mean_squared_error', cv=5) 

rmse_scores = (-scores)**0.5
 
rmse_mean = np.mean(rmse_scores)
rmse_std = np.std(rmse_scores)
 
sem = rmse_std / np.sqrt(len(rmse_scores))

# 95% confidence interval
z = 1.96  
lower_bound = rmse_mean - z * sem
upper_bound = rmse_mean + z * sem

# Print results
print(f"Cross-Validated RMSE: {rmse_mean:.2f}")
print(f"95% Confidence Interval: [{lower_bound:.2f}, {upper_bound:.2f}]")

Cross-Validated RMSE: 26.56
95% Confidence Interval: [26.23, 26.89]


## XGB

In [47]:
from xgboost import XGBRegressor
import xgboost as XGB

In [14]:
xgb = XGBRegressor( objective="reg:squarederror", tree_method="hist", # Fast and accurate for CPU 
                    random_state=42, n_jobs=-1 )

In [19]:
param_grid = { "n_estimators": [200, 500, 800], # Boosting rounds
               "max_depth": [4, 5,6, 7,8,9], # Tree depth 
               "learning_rate": [0.01, 0.05, 0.1], "subsample": [0.8, 1.0], # Row sampling
               "colsample_bytree": [0.6, 0.8, 1.0], # Column sampling
               "eta": [0.01,0.03,0.05]
             }

In [20]:
from sklearn.model_selection import KFold
kfold = KFold(n_splits=5 , random_state=42,shuffle=True)

In [21]:
grid = GridSearchCV( estimator=xgb, param_grid=param_grid, scoring="neg_root_mean_squared_error", # ✅ RMSE scoring 
                     cv=kfold, verbose=1, n_jobs=-1 )

In [22]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 972 candidates, totalling 4860 fits


KeyboardInterrupt: 

In [ ]:
best_xgb = grid.best_estimator_ 
print("Best Params:", grid.best_params_) 
print("CV RMSE:", -grid.best_score_)

In [44]:
params = {'n_estimators': 350, 'learning_rate': 0.011063400853562333, 'max_depth': 3,
 'min_child_weight': 10, 'gamma': 4.514784897326459, 'subsample': 0.6462218094725269, 
 'colsample_bytree': 0.7514562925918475, 'reg_alpha': 4.898472548846284, 'reg_lambda': 1.9769692203596296}

In [49]:
features = list(dv.get_feature_names_out())
dtrain = XGB.DMatrix(X_train, label=y_train, feature_names=features)

model = XGB.train(params, dtrain, num_boost_round=168, evals=[(dtrain, 'train'),], verbose_eval=5, early_stopping_rounds=5)

[0]	train-rmse:26.55955
[5]	train-rmse:26.55162
[10]	train-rmse:26.54214
[15]	train-rmse:26.53516
[20]	train-rmse:26.52853
[25]	train-rmse:26.52122
[30]	train-rmse:26.51611
[35]	train-rmse:26.50735


C:\Users\AJAY\miniconda3\Lib\site-packages\xgboost\callback.py:386: UserWarning: [22:56:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()


[40]	train-rmse:26.50065
[45]	train-rmse:26.49341
[50]	train-rmse:26.48628
[55]	train-rmse:26.47848
[60]	train-rmse:26.47201
[65]	train-rmse:26.46565
[70]	train-rmse:26.45832
[75]	train-rmse:26.45121
[80]	train-rmse:26.44536
[85]	train-rmse:26.43925
[90]	train-rmse:26.43239
[95]	train-rmse:26.42631
[100]	train-rmse:26.41989
[105]	train-rmse:26.41321
[110]	train-rmse:26.40673
[115]	train-rmse:26.40063
[120]	train-rmse:26.39362
[125]	train-rmse:26.38843
[130]	train-rmse:26.38218
[135]	train-rmse:26.37662
[140]	train-rmse:26.37054
[145]	train-rmse:26.36503
[150]	train-rmse:26.35971
[155]	train-rmse:26.35365
[160]	train-rmse:26.34784
[165]	train-rmse:26.34251
[167]	train-rmse:26.33997


In [52]:
X_val = XGB.DMatrix(X_val, feature_names=features)
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred)) 
print("Test RMSE:", rmse)

Test RMSE: 26.43498806033398


In [ ]:
# RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import optuna
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

num_cols = ['RhythmScore', 'AudioLoudness', 'VocalContent', 'AcousticQuality',
       'InstrumentalScore', 'LivePerformanceLikelihood', 'MoodScore',
       'TrackDurationMs', 'Energy']
preprocess = ColumnTransformer([('num', StandardScaler(), num_cols)], remainder='passthrough')

def tune_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 800)
    max_depth = trial.suggest_int('max_depth', 5, 25)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    
    model = Pipeline([
    ('prep', preprocess),
    ('rf', RandomForestRegressor(
    n_estimators=n_estimators,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    random_state=42
    ))
     ])
    score = cross_val_score(model, df_train, y_train, scoring='neg_root_mean_squared_error', cv=3).mean()
    return -score

study = optuna.create_study(direction='minimize')
study.optimize(tune_rf, n_trials=30)
study.best_params

[I 2025-11-19 23:11:25,936] A new study created in memory with name: no-name-424ebbfd-0710-431e-b7c6-41000117ef6f
[I 2025-11-19 23:11:55,044] Trial 0 finished with value: 26.69683990390641 and parameters: {'n_estimators': 159, 'max_depth': 15, 'min_samples_split': 16, 'min_samples_leaf': 4}. Best is trial 0 with value: 26.69683990390641.
